In [0]:
%sql
-- ============================================================================
-- Tabela......: mart_pagamento_credito
-- Camada......: Gold (Data Product)
--
-- Objetivo
-- ----------------------------------------------------------------------------
-- Consolidar mensalmente os principais indicadores operacionais de cobrança
-- e comportamento de pagamentos.
--
-- Perguntas de negócio
-- ----------------------------------------------------------------------------
-- • Estamos recebendo as parcelas em dia?
-- • Qual o índice de recebimento?
-- • Quantas parcelas permanecem em aberto?
-- • Quanto ainda falta receber?
-- • Os atrasos estão aumentando?
-- • Como evolui o comportamento dos pagamentos ao longo do tempo?
--
-- Grão
-- ----------------------------------------------------------------------------
-- 1 linha por competência (dt_competencia)
--
-- Fonte
-- ----------------------------------------------------------------------------
-- gold.fact_parcela_credito
-- ============================================================================

CREATE OR REPLACE TABLE dev_credit_risk.gold.mart_pagamento_credito

USING DELTA

AS

SELECT

    ----------------------------------------------------------------------------
    -- Competência
    ----------------------------------------------------------------------------

    dt_competencia,

    ----------------------------------------------------------------------------
    -- Volume
    ----------------------------------------------------------------------------

    COUNT(*)                                        AS qtd_parcelas,

    SUM(flag_pagamento)                             AS parcelas_pagas,

    SUM(
        CASE
            WHEN flag_pagamento = 0 THEN 1
            ELSE 0
        END
    )                                               AS parcelas_em_aberto,

    ----------------------------------------------------------------------------
    -- Financeiro
    ----------------------------------------------------------------------------

    SUM(valor_parcela)                              AS valor_esperado,

    SUM(COALESCE(valor_pago,0))                     AS valor_recebido,

    SUM(valor_em_aberto)                            AS saldo_em_aberto,

    ----------------------------------------------------------------------------
    -- Indicadores
    ----------------------------------------------------------------------------

    ROUND(

        SUM(COALESCE(valor_pago,0))
        /
        NULLIF(SUM(valor_parcela),0)

    ,4)                                             AS indice_recebimento,

    ROUND(

        SUM(
            CASE
                WHEN flag_pagamento = 1
                 AND dias_atraso <= 0
                THEN 1
                ELSE 0
            END
        )
        /
        NULLIF(SUM(flag_pagamento),0)

    ,4)                                             AS tx_pagamento_em_dia,

    ROUND(

        SUM(flag_pagamento_atrasado)
        /
        COUNT(*)

    ,4)                                             AS tx_pagamento_atrasado,

    ROUND(

        SUM(flag_pagamento_default_90)
        /
        COUNT(*)

    ,4)                                             AS tx_default,

    AVG(

        CASE
            WHEN flag_pagamento = 1
            THEN dias_atraso
        END

    )                                               AS media_dias_atraso,

    ----------------------------------------------------------------------------
    -- Auditoria
    ----------------------------------------------------------------------------

    current_timestamp()                             AS dt_carga

FROM dev_credit_risk.gold.fact_parcela_credito

GROUP BY

    dt_competencia

ORDER BY

    dt_competencia;

In [0]:
%sql
select * from dev_credit_risk.gold.mart_pagamento_credito
